In [1]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers import pipeline

I0000 00:00:1785786761.296604   28719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785786761.328737   28719 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785786762.072863   28719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
def analyser(path):

    
# 1. Chargement et traitement des fichiers audios
    
    # Chargement du fichier audio
    waveform, sample_rate = torchaudio.load(path)
    
    # Conversion du fichier en mono, si stéréo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    # Modification de la fréquene de l'audio pour correspondre au modèle utilisé
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    
    # Conversion de l'audio en tenseur pytorch
    input_values = processor(waveform.squeeze(), return_tensors="pt", sampling_rate=16000).input_values

    
# 2. Transcription vocale en texte
    
    # Transcription
    with torch.no_grad():
        logits = model(input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0])

    
# 3. Analyse de sentiment
    
    # Initialisation de l'analyseur de sentiment avec BERT
    sentiment_analyzer = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    result = sentiment_analyzer(transcription)

    
# 4. Classification de la transcription
    
    # Affichage du sentiment détecté
    if "1" in result[0]["label"] or "2" in result[0]["label"]:
        return "Sentiment négatif."
    elif "3" in result[0]["label"]:
        return "Sentiment neutre."
    else:
        return "Sentiment positif."

In [4]:
import gradio as gr

gr.Interface(fn=analyser, inputs="file", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0
